# Training Analysis — Learning Curves & Reward Progression

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

from src.environment.trading_env import TradingEnv
from src.agents.dqn_agent import build_agent, load_agent, run_episode
from src.training.trainer import train
from src.evaluation.metrics import compute_all
from src.evaluation.backtester import run_all, compare_strategies

with open("../configs/dqn_config.yaml") as f:
    cfg = yaml.safe_load(f)

ENV_CFG   = cfg["environment"]
MODEL_CFG = cfg["model"]
TRAIN_CFG = cfg["training"]

df_train_norm = pd.read_csv("../data/normalized/train.csv", index_col=0, parse_dates=True)
df_val_norm   = pd.read_csv("../data/normalized/val.csv",   index_col=0, parse_dates=True)
df_train      = pd.read_csv("../data/features/train.csv",   index_col=0, parse_dates=True)

print("Config:", yaml.dump({"model": MODEL_CFG, "training": TRAIN_CFG}, default_flow_style=False))

## 1. Train the DQN Agent

In [ ]:
os.makedirs("../models/best",         exist_ok=True)
os.makedirs("../models/checkpoints",  exist_ok=True)
os.makedirs("../results/logs",        exist_ok=True)

env_train = TradingEnv(df_train_norm, ENV_CFG)
env_val   = TradingEnv(df_val_norm,   ENV_CFG)

agent = build_agent(env_train, MODEL_CFG)

# Patch training config paths to point one level up
train_cfg = {**TRAIN_CFG,
             "best_model_save_path": "../models/best",
             "checkpoint_save_path": "../models/checkpoints",
             "log_path":             "../results/logs"}

agent = train(env_train, env_val, agent, train_cfg)
print("Training complete.")

## 2. Learning Curve — Validation Reward Over Training Steps

In [ ]:
# EvalCallback saves evaluations to results/logs/evaluations.npz
eval_log = Path("../results/logs/evaluations.npz")
if eval_log.exists():
    data = np.load(eval_log)
    timesteps = data["timesteps"]
    results_arr = data["results"]     # shape (n_evals, n_eval_episodes)
    mean_reward = results_arr.mean(axis=1)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(timesteps, mean_reward, marker="o", markersize=4, linewidth=1.5)
    ax.axhline(0, color="grey", linewidth=1, linestyle=":")
    ax.set_xlabel("Training Timesteps")
    ax.set_ylabel("Mean Val Episode Reward")
    ax.set_title("DQN Learning Curve — Validation Reward vs Timesteps")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("../results/figures/04_learning_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Best val reward : {mean_reward.max():.6f}  at step {timesteps[mean_reward.argmax()]:,}")
else:
    print("No evaluations.npz found yet — run the training cell above first.")

## 3. DQN Action Distribution on Validation Set

In [ ]:
# Load best model and run on val set
best_model_path = "../models/best/best_model"
if Path(best_model_path + ".zip").exists():
    env_val2 = TradingEnv(df_val_norm, ENV_CFG)
    best_agent = load_agent(best_model_path, env_val2)
    val_trace = run_episode(best_agent, env_val2)

    action_counts = val_trace["action"].value_counts().sort_index()
    action_labels = {0: "Hold", 1: "Buy", 2: "Sell"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Action bar chart
    axes[0].bar([action_labels[a] for a in action_counts.index], action_counts.values,
                color=["grey", "green", "red"])
    axes[0].set_title("Action Distribution — Validation Set (Best Model)")
    axes[0].set_ylabel("Count")
    axes[0].grid(axis="y", alpha=0.3)

    # Portfolio value on val
    axes[1].plot(val_trace.index, val_trace["portfolio_value"], linewidth=2, label="DQN")
    axes[1].axhline(ENV_CFG["initial_balance"], color="grey", linewidth=1, linestyle=":", label="Initial")
    axes[1].set_title("DQN Portfolio Value — Validation Set")
    axes[1].set_ylabel("Portfolio Value")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("../results/figures/04_dqn_val_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

    val_metrics = compute_all(val_trace["portfolio_value"])
    print("DQN validation metrics:")
    for k, v in val_metrics.items():
        print(f"  {k:<22s}: {v:+.4f}")
else:
    print("Best model not found — run the training cell first.")